# 港铁线网拓扑图可视化 (MTR Topology Visualization)

本 Notebook 用于加载预处理好的 `.gml` 网络文件，并利用 `NetworkX` 和 `Plotly` 进行交互式的图表展示。
此文件位于本项目的 `data/processed/mtr_topology.gml`，包含了同一站点的跨线剥离逻辑（Node Splitting）连线以及物理的同线运行区间连线。

In [5]:
# 导入相关库
import networkx as nx
import plotly.graph_objects as go
import os

In [6]:
# 加载 MTR 拓扑图数据
gml_path = r"E:\CityU_CS\CS 5483\CS5483_Team-Project_MTR-Stress-Flow-Simulation\data\processed\mtr_topology.gml"

# 使用 NetworkX 读取 GML 格式文件
if not os.path.exists(gml_path):
    print("找不到指定的 GML 文件，请确保网络拓扑已正确生成。")
else:
    G = nx.read_gml(gml_path)
    print(f"图加载成功：共包含 {G.number_of_nodes()} 个节点，{G.number_of_edges()} 条连线。")

图加载成功：共包含 120 个节点，272 条连线。


In [7]:
# 计算布局并绘图 (使用弹簧引力布局 Spring Layout)
pos = nx.spring_layout(G, k=0.3, iterations=100, seed=42)

running_edge_x, running_edge_y = [], []
transfer_edge_x, transfer_edge_y = [], []

# 分离两种类型的边缘 (Transfer 和 Running) 去区别化展示
for edge in G.edges(data=True):
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_type = edge[2].get('edge_type', 'unknown')
    
    if edge_type == 'transfer':
        transfer_edge_x.extend([x0, x1, None])
        transfer_edge_y.extend([y0, y1, None])
    else:
        running_edge_x.extend([x0, x1, None])
        running_edge_y.extend([y0, y1, None])

# 定义轨道线样式
running_edge_trace = go.Scatter(
    x=running_edge_x, y=running_edge_y,
    line=dict(width=2, color='#333'),
    hoverinfo='none',
    mode='lines',
    name='区间行车线 (Running)'
)

# 定义站内换乘线样式
transfer_edge_trace = go.Scatter(
    x=transfer_edge_x, y=transfer_edge_y,
    line=dict(width=1.5, color='#aaa', dash='dot'),
    hoverinfo='none',
    mode='lines',
    name='站内换乘线 (Transfer)'
)

# 计算并记录节点位置、悬停文本及颜色
node_colors, node_text, node_hovertext, node_x, node_y = [], [], [], [], []

# 港铁官方线路颜色
mtr_colors = {
    'KTL': '#00ab4e', # 观塘线
    'TWL': '#ed1d24', # 荃湾线
    'ISL': '#0071ce', # 港岛线
    'SIL': '#b6bd00', # 南港岛线
    'TKL': '#a35eb5', # 将军澳线
    'TCL': '#f7943f', # 东涌线
    'DRL': '#f173ac', # 迪士尼线
    'AEL': '#00888a', # 机场快线
    'EAL': '#53b7e8', # 东铁线
    'TML': '#8d6019'  # 屯马线
}

for node in G.nodes():
    node_data = G.nodes[node]
    line = node_data.get('line', 'Unknown')
    
    # 分配官方颜色，匹配不到的用默认灰色
    base_line = line.split('_')[0] if '_' in line else line
    node_colors.append(mtr_colors.get(base_line, '#888888'))
    
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    
    # 信息标签
    c_name = node_data.get('c_name', '')
    e_name = node_data.get('name', '')
    node_hovertext.append(f"Node ID: {node}<br>Station: {c_name} ({e_name})<br>Line: {line}")
    node_text.append(c_name)

# 站点绘制设置
node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text', hoverinfo='text',
    text=node_text, textposition="top center", hovertext=node_hovertext,
    textfont=dict(size=9, color="#444"), 
    marker=dict(showscale=False, color=node_colors, size=12, line_width=1.5, line_color="white"),
    name='站台节点'
)

# 渲染完整图表
fig = go.Figure(
    data=[running_edge_trace, transfer_edge_trace, node_trace],
    layout=go.Layout(
        title=dict(text='MTR Network Topology Diagram (Official Colors)', font=dict(size=18)),
        showlegend=True, 
        hovermode='closest',
        margin=dict(b=20, l=5, r=5, t=50), 
        paper_bgcolor="#ffffff",
        plot_bgcolor="#ffffff",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        width=1000, 
        height=800
    )
)

# 展现图表
fig.show()